In [1]:
# GOAL : make sample info file with relative information

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from pandas_plink import read_grm
import sys; sys.path.append("/data/jerrylee/pjt/BIGFAM.v.0.1")
from BIGFAM.pedigree import ped
import importlib

# UKB

## Step 1. Load sample info

In [2]:
# [Arguments]
basic_info_path = '/data/jerrylee/data/UKB/pheno/pheno_basic' 
# basic_info includes [sex, age, ethnicitiy, kinship, PCA]
eur_info_fn = "/data02/jerrylee/xChrom/data/info_family/sample.eur.info"
kinship_fn = "/data01/UKB202005/data/genotype/ukbgene_rel/ukb59688_rel_s488249.dat"
fam_fn = '/data/jerrylee/data/UKB/geno/ukb_chr{}.qc.fam'

In [3]:
# load files
df_info_raw = pd.read_csv(eur_info_fn, sep='\t')
df_kinship_raw = pd.read_csv(kinship_fn, sep='\s+')

# make kinship long (concat with flipped dataframe)
df_kinship_flip = df_kinship_raw.copy()
df_kinship_flip.columns = ["ID2" ,"ID1", "HetHet", "IBS0", "Kinship"]
df_kinship = pd.concat([df_kinship_raw, df_kinship_flip]).reset_index(drop=True)
df_kinship.columns = ["eid", "eid_rel", "HetHet", "IBS0", "Kinship"]

# merge with EUR info
df_info = df_info_raw[["eid", "age", "sex"]]
df_tmp = pd.merge(df_info, df_kinship, on=["eid"])
df_info_tmp = df_info.copy()
df_info_tmp.columns = ["eid_rel", "age_rel", "sex_rel"]
df_merged = pd.merge(df_tmp, df_info_tmp, on=["eid_rel"])
df_merged["kinship"] = 2 * df_merged["Kinship"]

n_unique_eid = len(df_merged["eid"].unique())
print(n_unique_eid)
df_merged

142217


,eid,age,sex,eid_rel,HetHet,IBS0,Kinship,age_rel,sex_rel,kinship
0,1000031,52.0,0.0,4977365,0.046,0.0139,0.0684,45.0,0.0,0.1368
1,4207488,62.0,1.0,4977365,0.044,0.0154,0.0474,45.0,0.0,0.0948
2,1000073,54.0,1.0,1468150,0.047,0.0131,0.0753,60.0,0.0,0.1506
3,1000094,65.0,0.0,3653174,0.077,0.0052,0.2471,64.0,0.0,0.4942
4,1000122,68.0,0.0,1390862,0.051,0.0094,0.1215,63.0,0.0,0.2430
...,...,...,...,...,...,...,...,...,...,...
205987,6024624,47.0,0.0,3342229,0.044,0.0154,0.0497,67.0,1.0,0.0994
205988,6024638,68.0,1.0,5832823,0.085,0.0026,0.2970,64.0,1.0,0.5940
205989,6024646,46.0,0.0,1244265,0.045,0.0147,0.0589,42.0,1.0,0.1178
205990,6024769,69.0,0.0,4102199,0.067,0.0000,0.2541,44.0,0.0,0.5082


## Step 2. Add `DOR` annotation using kinship coefficient

In [5]:
# ad-hoc functions
def get_unique_eid(df):
    """Count unique `eid` from given dataframe."""
    eids = set(df["eid"])
    eid_rels = set(df["eid_rel"])
    return list(eids | eid_rels)

def save_relation(df, path, fn):
    df_tmp = (df[["eid", "age", "sex", "eid_rel", "age_rel", "sex_rel", "Kinship"]]
              .astype({"age": int, "sex": int, "age_rel": int, "sex_rel": int})
    )
    df_tmp.to_csv(path + "/" + fn, sep='\t', index=False)
    
def drop_dupicate_eid(df):
    "left unique eid pairs (remove swapped eid-eid_rel pairs)"
    tmp = df.copy()
    tmp['key'] = tmp.apply(lambda row: map(str, sorted([row['eid'], row['eid_rel']])), axis=1)
    tmp["key"] = tmp.apply(lambda row: "_".join(row["key"]), axis=1)
    
    return tmp.drop_duplicates(subset="key").drop(columns="key")

In [6]:
# Check Degree of Relativeness (DOR)
# number of sample in each relatives -- check
kinship_coefficients = {1: 0.5,
                        2: 0.5 / 2,
                        3: 0.5 / 4,
                        4: 0.5 / 8}

for dor, kinship_coeff in kinship_coefficients.items():
    lower_lim = 0.8 * kinship_coeff
    upper_lim = 1.2 * kinship_coeff
    is_kinship_near_coeff = (df_merged["kinship"] > lower_lim) & (df_merged["kinship"] < upper_lim)
    df_merged.loc[is_kinship_near_coeff, "DOR"] = dor
    print(
        "{} DOR (n_pair = {}, n={}): kinship coefficient within ({:.3f}, {:.3f})".format(
        dor,
        int(df_merged[df_merged["DOR"] == dor].shape[0] / 2),
        len(get_unique_eid(df_merged[df_merged["DOR"] == dor])),
        lower_lim,
        upper_lim
    ),
          flush=True)
    

1 DOR (n_pair = 27365, n=49083): kinship coefficient within (0.400, 0.600)
2 DOR (n_pair = 9447, n=17240): kinship coefficient within (0.200, 0.300)
3 DOR (n_pair = 44514, n=72944): kinship coefficient within (0.100, 0.150)
4 DOR (n_pair = 0, n=0): kinship coefficient within (0.050, 0.075)


In [8]:
df = (df_merged[["DOR", "eid", "age", "sex", "eid_rel", "age_rel", "sex_rel", "kinship"]]
      .dropna())
df

,DOR,eid,age,sex,eid_rel,age_rel,sex_rel,kinship
0,3.0,1000031,52.0,0.0,4977365,45.0,0.0,0.1368
3,1.0,1000094,65.0,0.0,3653174,64.0,0.0,0.4942
4,2.0,1000122,68.0,0.0,1390862,63.0,0.0,0.2430
5,2.0,1000174,69.0,0.0,3765836,45.0,0.0,0.2504
6,3.0,1950469,41.0,1.0,3765836,45.0,0.0,0.1022
...,...,...,...,...,...,...,...,...
205986,2.0,6024607,43.0,1.0,4730713,63.0,0.0,0.2502
205988,1.0,6024638,68.0,1.0,5832823,64.0,1.0,0.5940
205989,3.0,6024646,46.0,0.0,1244265,42.0,1.0,0.1178
205990,1.0,6024769,69.0,0.0,4102199,44.0,0.0,0.5082


## Step 3. add X correlation

In [9]:
df["xcor"] = np.nan

# add sex_type column
df["sex_type"] = np.nan

is_sex = {"mm": df["sex"] + df["sex_rel"] == 2,
          "mf": df["sex"] + df["sex_rel"] == 1,
          "ff": df["sex"] + df["sex_rel"] == 0,}

for sex_type in ["mm", "mf", "ff"]:
    df.loc[is_sex[sex_type], "sex_type"] = sex_type
    
df.head()

,DOR,eid,age,sex,eid_rel,age_rel,sex_rel,kinship,xcor,sex_type
0,3.0,1000031,52.0,0.0,4977365,45.0,0.0,0.1368,NaN,ff
3,1.0,1000094,65.0,0.0,3653174,64.0,0.0,0.4942,NaN,ff
4,2.0,1000122,68.0,0.0,1390862,63.0,0.0,0.2430,NaN,ff
5,2.0,1000174,69.0,0.0,3765836,45.0,0.0,0.2504,NaN,ff
6,3.0,1950469,41.0,1.0,3765836,45.0,0.0,0.1022,NaN,mf


In [10]:
path = "/data/jerrylee/data/UKB/grm_rel"

### DOR1

In [11]:
# DOR1
rel_to_sp = {"FS": "mm", 
             "MS": "mf", 
             "FD": "mf", 
             "MD": "ff", 
             "SS": "mm", 
             "SD": "mf", 
             "DD": "ff"}

for rel in tqdm(rel_to_sp.keys()):
    is_cond = (df["DOR"] == 1) & (df["sex_type"] == rel_to_sp[rel])
    df_subset = df[is_cond]
    
    # load GRM
    filepath = f"{path}/dor1_grms/rel/{rel}.grm.bin"
    id_filepath = f"{path}/dor1_grms/rel/{rel}.grm.id"
    n_snps_filepath = f"{path}/dor1_grms/rel/{rel}.grm.N.bin"
    
    (K, _) = read_grm(filepath, id_filepath, n_snps_filepath)
    
    # for each pairs
    for index, row in df_subset.iterrows():
        id1, id2 = row[["eid", "eid_rel"]].values
        try:
            cr = float(K.loc[id1, id2].values)
            # add c_r
            this_rel = (df["eid"] == id1) & (df["eid_rel"] == id2)
            
            # check cryptic relatedness
            if sum(this_rel) > 1:
                print(id1, id2)
            df.loc[this_rel, "xcor"] = cr
        except:
            pass

100%|██████████| 7/7 [11:47<00:00, 101.04s/it]


### DOR2

In [13]:
# DOR2
for sex_type in tqdm(["mm", "mf", "ff"]):
    is_cond = (df["DOR"] == 2) & (df["sex_type"] == sex_type)
    df_subset = df[is_cond]
    
    # load GRM
    filepath = f"{path}/dor2_grms/dor2_{sex_type}.grm.bin"
    id_filepath = f"{path}/dor2_grms/dor2_{sex_type}.grm.id"
    n_snps_filepath = f"{path}/dor2_grms/dor2_{sex_type}.grm.N.bin"
    
    (K, _) = read_grm(filepath, id_filepath, n_snps_filepath)
    
    # for each pairs
    for _, row in df_subset.iterrows():
        id1, id2 = row[["eid", "eid_rel"]].values
        try:
            cr = float(K.loc[id1, id2].values)
            # add c_r
            this_rel = (df["eid"] == id1) & (df["eid_rel"] == id2)
            # check cryptic relatedness
            if sum(this_rel) > 1:
                print(id1, id2)
            df.loc[this_rel, "xcor"] = cr
        except:
            pass

100%|██████████| 3/3 [04:38<00:00, 92.76s/it] 


### DOR3

In [14]:
# DOR3
for sex_type in tqdm(["mm", "mf", "ff"]):
    is_cond = (df["DOR"] == 3) & (df["sex_type"] == sex_type)
    df_subset = df[is_cond]
    
    # load GRM
    filepath = f"{path}/dor3_grms/dor3_{sex_type}.grm.bin"
    id_filepath = f"{path}/dor3_grms/dor3_{sex_type}.grm.id"
    n_snps_filepath = f"{path}/dor3_grms/dor3_{sex_type}.grm.N.bin"
    
    (K, _) = read_grm(filepath, id_filepath, n_snps_filepath)
    
    # for each pairs
    for _, row in df_subset.iterrows():
        id1, id2 = row[["eid", "eid_rel"]].values
        try:
            cr = float(K.loc[id1, id2].values)
            # add c_r
            this_rel = (df["eid"] == id1) & (df["eid_rel"] == id2)
            # check cryptic relatedness
            if sum(this_rel) > 1:
                print(id1, id2)
            df.loc[this_rel, "xcor"] = cr
        except:
            pass

100%|██████████| 3/3 [22:40<00:00, 453.38s/it]


In [16]:
df

,DOR,eid,age,sex,eid_rel,age_rel,sex_rel,kinship,xcor,sex_type
0,3.0,1000031,52.0,0.0,4977365,45.0,0.0,0.1368,0.040435,ff
3,1.0,1000094,65.0,0.0,3653174,64.0,0.0,0.4942,0.632092,ff
4,2.0,1000122,68.0,0.0,1390862,63.0,0.0,0.2430,0.501473,ff
5,2.0,1000174,69.0,0.0,3765836,45.0,0.0,0.2504,0.332779,ff
6,3.0,1950469,41.0,1.0,3765836,45.0,0.0,0.1022,0.012638,mf
...,...,...,...,...,...,...,...,...,...,...
205986,2.0,6024607,43.0,1.0,4730713,63.0,0.0,0.2502,-0.012986,mf
205988,1.0,6024638,68.0,1.0,5832823,64.0,1.0,0.5940,0.386101,mm
205989,3.0,6024646,46.0,0.0,1244265,42.0,1.0,0.1178,0.719852,mf
205990,1.0,6024769,69.0,0.0,4102199,44.0,0.0,0.5082,0.472529,ff


### Step 3.1 Formatting dataframe

In [17]:
def check_coln(df):
    coln_type = {
        "eid": int, "eid_rel": int,
        "age": int, "age_rel": int,
        "sex": int, "sex_rel": int,
        "sex_type": str,
        "DOR": int, 
        "kinship": float, "xcor": float,
    }
    
    for coln, dtype in coln_type.items():
        assert coln in df.columns, f"No {coln} in dataframe."
        
        df[coln] = df[coln].astype(dtype)
    
    return df

In [28]:
df_relInfo = check_coln(df)
df_relInfo["age_diff"] = np.abs(df_relInfo["age"] - df_relInfo["age_rel"])
df_relInfo

,DOR,eid,age,sex,eid_rel,age_rel,sex_rel,kinship,xcor,sex_type,age_diff
0,3,1000031,52,0,4977365,45,0,0.1368,0.040435,ff,7
3,1,1000094,65,0,3653174,64,0,0.4942,0.632092,ff,1
4,2,1000122,68,0,1390862,63,0,0.2430,0.501473,ff,5
5,2,1000174,69,0,3765836,45,0,0.2504,0.332779,ff,24
6,3,1950469,41,1,3765836,45,0,0.1022,0.012638,mf,4
...,...,...,...,...,...,...,...,...,...,...,...
205986,2,6024607,43,1,4730713,63,0,0.2502,-0.012986,mf,20
205988,1,6024638,68,1,5832823,64,1,0.5940,0.386101,mm,4
205989,3,6024646,46,0,1244265,42,1,0.1178,0.719852,mf,4
205990,1,6024769,69,0,4102199,44,0,0.5082,0.472529,ff,25


In [29]:
# remove duplicated rows
df_relInfo['eid_tuple'] = df_relInfo[['eid', 'eid_rel']].apply(lambda x: tuple(sorted(x)), axis=1)

# Identify and drop rows with unique (eid, eid_rel)
unique_indices = ~df_relInfo.duplicated(subset='eid_tuple', keep='first')
df_sample = df_relInfo[unique_indices].drop(columns=['eid_tuple'])

## Step 4. Add `rel_type` annotation

## DOR1

In [30]:
# add relationship annotation
is_dor = df_sample["DOR"] == 1
is_sib = (df_sample["age_diff"] >= 1) & (df_sample["age_diff"] <= 5)
is_po = (df_sample["age_diff"] >= 20) & (df_sample["age_diff"] <= 28)

# male-male
is_mm = df_sample["sex_type"] == "mm"

exps = {"SB": 1/2,
        "FS": 0}

for rel, xcor in exps.items():
    if xcor == 0:
        is_rel = (df_sample["xcor"] >= -0.05) & (df_sample["xcor"] <= 0.05)
    else:
        is_rel = (df_sample["xcor"] >= 0.8*xcor) & (df_sample["xcor"] <= 1.2*xcor)

    if rel == "FS":
        df_sample.loc[is_dor & is_mm & is_po & is_rel, "rel_type"] = rel
        df_sample.loc[is_dor & is_mm & is_po & is_rel, "cx"] = xcor
    elif rel == "SB":
        df_sample.loc[is_dor & is_mm & is_sib & is_rel, "rel_type"] = rel
        df_sample.loc[is_dor & is_mm & is_sib & is_rel, "cx"] = xcor

# df_sample.loc[is_dor & is_mm & is_sib, "rel_type"] = "SB"
# df_sample.loc[is_dor & is_mm & is_sib, "cx"] = 1/2
# df_sample.loc[is_dor & is_mm & is_po, "rel_type"] = "FS"
# df_sample.loc[is_dor & is_mm & is_po, "cx"] = 0

# female-female
is_ff = df_sample["sex_type"] == "ff"

exps = {"MD": 1/2,
        "DS": 3/4}

for rel, xcor in exps.items():
    if xcor == 0:
        is_rel = (df_sample["xcor"] >= -0.05) & (df_sample["xcor"] <= 0.05)
    else:
        is_rel = (df_sample["xcor"] >= 0.8*xcor) & (df_sample["xcor"] <= 1.2*xcor)

    if rel == "MD":
        df_sample.loc[is_dor & is_ff & is_po & is_rel, "rel_type"] = rel
        df_sample.loc[is_dor & is_ff & is_po & is_rel, "cx"] = xcor
    elif rel == "DS":
        df_sample.loc[is_dor & is_ff & is_sib & is_rel, "rel_type"] = rel
        df_sample.loc[is_dor & is_ff & is_sib & is_rel, "cx"] = xcor
        
# df_sample.loc[is_dor & is_ff & is_sib, "rel_type"] = "DS"
# df_sample.loc[is_dor & is_ff & is_sib, "cx"] = 3/4
# df_sample.loc[is_dor & is_ff & is_po, "rel_type"] = "MD"
# df_sample.loc[is_dor & is_ff & is_po, "cx"] = 1/2

# male-female
is_mf = df_sample["sex_type"] == "mf"
is_female_older = ((df_sample["age"] > df_sample["age_rel"]) & (df_sample["sex"] == 0)) | ((df_sample["age"] < df_sample["age_rel"]) & (df_sample["sex_rel"] == 0))
is_female_younger = ((df_sample["age"] > df_sample["age_rel"]) & (df_sample["sex_rel"] == 0)) | ((df_sample["age"] < df_sample["age_rel"]) & (df_sample["sex"] == 0))

exps = {"SS_DB": np.sqrt(2)/4,
        "MS": np.sqrt(2)/2,
        "FD": np.sqrt(2)/2}

for rel, xcor in exps.items():
    if xcor == 0:
        is_rel = (df_sample["xcor"] >= -0.05) & (df_sample["xcor"] <= 0.05)
    else:
        is_rel = (df_sample["xcor"] >= 0.8*xcor) & (df_sample["xcor"] <= 1.2*xcor)

    if rel == "SS_DB":
        df_sample.loc[is_dor & is_mf & is_sib & is_rel, "rel_type"] = rel
        df_sample.loc[is_dor & is_mf & is_sib & is_rel, "cx"] = xcor
    elif rel == "MS":
        df_sample.loc[is_dor & is_mf & is_po & is_female_older & is_rel, "rel_type"] = rel
        df_sample.loc[is_dor & is_mf & is_po & is_female_older & is_rel, "cx"] = xcor
    elif rel == "FD":
        df_sample.loc[is_dor & is_mf & is_po & is_female_younger & is_rel, "rel_type"] = rel
        df_sample.loc[is_dor & is_mf & is_po & is_female_younger & is_rel, "cx"] = xcor
        
        
# df_sample.loc[is_dor & is_mf & is_sib, "rel_type"] = "SS_DB"
# df_sample.loc[is_dor & is_mf & is_sib, "cx"] = np.sqrt(2)/4

# is_female_older = ((df_sample["age"] > df_sample["age_rel"]) & (df_sample["sex"] == 0)) | ((df_sample["age"] < df_sample["age_rel"]) & (df_sample["sex_rel"] == 0))
# is_female_younger = ((df_sample["age"] > df_sample["age_rel"]) & (df_sample["sex_rel"] == 0)) | ((df_sample["age"] < df_sample["age_rel"]) & (df_sample["sex"] == 0))
# df_sample.loc[is_dor & is_mf & is_po & is_female_older, "rel_type"] = "MS" # FD or MS
# df_sample.loc[is_dor & is_mf & is_po & is_female_older, "cx"] = np.sqrt(2)/2
# df_sample.loc[is_dor & is_mf & is_po & is_female_younger, "rel_type"] = "FD"
# df_sample.loc[is_dor & is_mf & is_po & is_female_younger, "cx"] = np.sqrt(2)/2

In [37]:
for_sums = df_sample[is_dor]

tmp_sums = (for_sums[["rel_type", "age_diff", "kinship", "xcor", "cx"]]
            .dropna(subset=["rel_type"])
            .groupby("rel_type")
            .mean()
            .round(3))
tmp_size = for_sums.dropna(subset=["rel_type"]).groupby("rel_type").size()
pd.merge(pd.DataFrame(tmp_size, columns=["size"]), 
         tmp_sums,
         on="rel_type")

,size,age_diff,kinship,xcor,cx
rel_type,,,,,
DS,3137,2.582,0.498,0.743,0.750
FD,949,24.335,0.497,0.680,0.707
FS,637,24.425,0.496,-0.000,0.000
MD,1874,23.776,0.497,0.485,0.500
MS,1281,23.754,0.496,0.678,0.707
SB,665,2.615,0.498,0.497,0.500
SS_DB,1861,3.012,0.499,0.354,0.354


## DOR2

In [38]:
is_dor = df_sample["DOR"] == 2

# add relationship annotation
is_parent_s_sib = (df_sample["age_diff"] >= 15) & (df_sample["age_diff"] <= 29)

is_mm = df_sample["sex_type"] == "mm"
exps = {"SFB": 0,
        "SMB": 1/4}

for rel, xcor in exps.items():
    if xcor == 0:
        is_rel = (df_sample["xcor"] >= -0.05) & (df_sample["xcor"] <= 0.05)
    else:
        is_rel = (df_sample["xcor"] >= 0.8*xcor) & (df_sample["xcor"] <= 1.2*xcor)

    df_sample.loc[is_dor & is_mm & is_parent_s_sib & is_rel, "rel_type"] = rel
    df_sample.loc[is_dor & is_mm & is_parent_s_sib & is_rel, "cx"] = xcor


# female-female
is_ff = df_sample["sex_type"] == "ff"
exps = {"DFS": 1/4,
        "DMS": 3/8}

for rel, xcor in exps.items():
    if xcor == 0:
        is_rel = (df_sample["xcor"] >= -0.05) & (df_sample["xcor"] <= 0.05)
    else:
        is_rel = (df_sample["xcor"] >= 0.8*xcor) & (df_sample["xcor"] <= 1.2*xcor)

    df_sample.loc[is_dor & is_ff & is_parent_s_sib & is_rel, "rel_type"] = rel
    df_sample.loc[is_dor & is_ff & is_parent_s_sib & is_rel, "cx"] = xcor

# male-female
is_mf = df_sample["sex_type"] == "mf"
exp_SFS = 0
exp_SMS = 3 * np.sqrt(2) / 8
exp_DFB = np.sqrt(2) / 4
exp_DMB = np.sqrt(2) / 8

is_SFS = (df_sample["xcor"] >= 0.8*exp_SFS) & (df_sample["xcor"] <= 1.2*exp_SFS)
is_SMS = (df_sample["xcor"] >= 0.8*exp_SMS) & (df_sample["xcor"] <= 1.2*exp_SMS)
is_DFB = (df_sample["xcor"] >= 0.8*exp_DFB) & (df_sample["xcor"] <= 1.2*exp_DFB)
is_DMB = (df_sample["xcor"] >= 0.8*exp_DMB) & (df_sample["xcor"] <= 1.2*exp_DMB)

is_female_older = ((df_sample["age"] > df_sample["age_rel"]) & (df_sample["sex"] == 0)) | ((df_sample["age"] < df_sample["age_rel"]) & (df_sample["sex_rel"] == 0))
is_female_younger = ((df_sample["age"] > df_sample["age_rel"]) & (df_sample["sex_rel"] == 0)) | ((df_sample["age"] < df_sample["age_rel"]) & (df_sample["sex"] == 0))

df_sample.loc[is_dor & is_mf & is_parent_s_sib & is_SFS & is_female_older, "rel_type"] = "SFS"
df_sample.loc[is_dor & is_mf & is_parent_s_sib & is_SFS & is_female_older, "cx"] = exp_SFS

df_sample.loc[is_dor & is_mf & is_parent_s_sib & is_SMS & is_female_older, "rel_type"] = "SMS"
df_sample.loc[is_dor & is_mf & is_parent_s_sib & is_SMS & is_female_older, "cx"] = exp_SMS

df_sample.loc[is_dor & is_mf & is_parent_s_sib & is_DFB & is_female_younger, "rel_type"] = "DFB"
df_sample.loc[is_dor & is_mf & is_parent_s_sib & is_DFB & is_female_younger, "cx"] = exp_DFB

df_sample.loc[is_dor & is_mf & is_parent_s_sib & is_DMB & is_female_younger, "rel_type"] = "DMB"
df_sample.loc[is_dor & is_mf & is_parent_s_sib & is_DMB & is_female_younger, "cx"] = exp_DMB

In [39]:
for_sums = df_sample[is_dor]

tmp_sums = (for_sums[["rel_type", "age_diff", "kinship", "xcor", "cx"]]
            .dropna(subset=["rel_type"])
            .groupby("rel_type")
            .mean()
            .round(3))
tmp_size = for_sums.dropna(subset=["rel_type"]).groupby("rel_type").size()
pd.merge(pd.DataFrame(tmp_size, columns=["size"]), 
         tmp_sums,
         on="rel_type")

,size,age_diff,kinship,xcor,cx
rel_type,,,,,
DFB,286,20.052,0.249,0.346,0.354
DFS,330,20.206,0.249,0.252,0.250
DMB,229,20.310,0.249,0.175,0.177
DMS,579,20.352,0.248,0.378,0.375
SFB,586,20.444,0.247,-0.000,0.000
SMB,98,20.378,0.250,0.246,0.250
SMS,346,20.049,0.251,0.538,0.530


## DOR3

In [40]:
is_dor = df_sample["DOR"] == 3
# add relationship annotation
is_offlevel = (df_sample["age_diff"] >= 0) & (df_sample["age_diff"] <= 5)

# male-male
is_mm = df_sample["sex_type"] == "mm"
exps = {"SFBS_SFSS_SMBS": 0,
        "SMSS": 3/8}

for rel, xcor in exps.items():
    if xcor == 0:
        is_rel = (df_sample["xcor"] >= -0.05) & (df_sample["xcor"] <= 0.05)
    else:
        is_rel = (df_sample["xcor"] >= 0.8*xcor) & (df_sample["xcor"] <= 1.2*xcor)

    df_sample.loc[is_dor & is_mm & is_offlevel & is_rel, "rel_type"] = rel
    df_sample.loc[is_dor & is_mm & is_offlevel & is_rel, "cx"] = xcor


# female-female
is_ff = df_sample["sex_type"] == "ff"
exps = {"DFBD_DMBD": 1/4,
        "DFSD": 1/8,
        "DMSD": 3/16}

for rel, xcor in exps.items():
    if xcor == 0:
        is_rel = (df_sample["xcor"] >= -0.05) & (df_sample["xcor"] <= 0.05)
    else:
        is_rel = (df_sample["xcor"] >= 0.8*xcor) & (df_sample["xcor"] <= 1.2*xcor)

    df_sample.loc[is_dor & is_ff & is_offlevel & is_rel, "rel_type"] = rel
    df_sample.loc[is_dor & is_ff & is_offlevel & is_rel, "cx"] = xcor


# male-female
is_mf = df_sample["sex_type"] == "mf"

exps = {"SFBD_SFSD_DFBS_DMBS": 0,
        "SMBD_DFSS" : np.sqrt(2)/8,
        "SMSD_DMSS" : 3*np.sqrt(2)/16,}

for rel, xcor in exps.items():
    if xcor == 0:
        is_rel = (df_sample["xcor"] >= -0.05) & (df_sample["xcor"] <= 0.05)
    else:
        is_rel = (df_sample["xcor"] >= 0.8*xcor) & (df_sample["xcor"] <= 1.2*xcor)

    df_sample.loc[is_dor & is_mf & is_offlevel & is_rel, "rel_type"] = rel
    df_sample.loc[is_dor & is_mf & is_offlevel & is_rel, "cx"] = xcor


In [41]:
for_sums = df_sample[is_dor]

tmp_sums = (for_sums[["rel_type", "age_diff", "kinship", "xcor", "cx"]]
            .dropna(subset=["rel_type"])
            .groupby("rel_type")
            .mean()
            .round(3))
tmp_size = for_sums.dropna(subset=["rel_type"]).groupby("rel_type").size()
pd.merge(pd.DataFrame(tmp_size, columns=["size"]), 
         tmp_sums,
         on="rel_type")

,size,age_diff,kinship,xcor,cx
rel_type,,,,,
DFBD_DMBD,921,2.494,0.125,0.261,0.250
DFSD,935,2.536,0.124,0.125,0.125
DMSD,1274,2.572,0.124,0.186,0.188
SFBD_SFSD_DFBS_DMBS,5970,2.560,0.124,0.001,0.000
SFBS_SFSS_SMBS,3369,2.588,0.124,-0.001,0.000
SMBD_DFSS,947,2.571,0.124,0.176,0.177
SMSD_DMSS,1051,2.523,0.124,0.263,0.265
SMSS,287,2.700,0.124,0.372,0.375


In [43]:
df_sample.dropna()

,DOR,eid,age,sex,eid_rel,age_rel,sex_rel,kinship,xcor,sex_type,age_diff,rel_type,cx
3,1,1000094,65,0,3653174,64,0,0.4942,0.632092,ff,1,DS,0.750000
5,2,1000174,69,0,3765836,45,0,0.2504,0.332779,ff,24,DMS,0.375000
6,3,1950469,41,1,3765836,45,0,0.1022,0.012638,mf,4,SFBD_SFSD_DFBS_DMBS,0.000000
9,3,1950469,41,1,4593489,42,1,0.1154,-0.000106,mm,1,SFBS_SFSS_SMBS,0.000000
24,3,1000243,64,1,2489078,62,1,0.1070,0.049882,mm,2,SFBS_SFSS_SMBS,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
203475,1,5903106,52,0,5935693,55,0,0.5042,0.609106,ff,3,DS,0.750000
203680,3,5912328,58,0,5959541,58,1,0.1290,0.296907,mf,0,SMSD_DMSS,0.265165
203747,3,5915997,69,1,5970575,69,1,0.1154,-0.042444,mm,0,SFBS_SFSS_SMBS,0.000000
204312,1,5943063,42,0,6007495,64,1,0.4972,0.617095,mf,22,FD,0.707107


## Step 5. save results

In [55]:
path_to_save = "/home/jerrylee/data/pjt/BIGFAM.v.0.1/data/UKB/relative_information"

In [45]:
def check_coln(df):
    coln_type = {
        "eid": int, "eid_rel": int,
        "age": int, "age_rel": int,
        "sex": int, "sex_rel": int,
        "sex_type": str, "rel_type": str,
        "DOR": int, 
        "kinship": float, 
        "xcor": float, "cx": float,
    }
    
    for coln, dtype in coln_type.items():
        assert coln in df.columns, f"No {coln} in dataframe."
        
        df[coln] = df[coln].astype(dtype)
    
    return df

In [71]:
df_to_save = (check_coln(df_sample)
              .rename(columns={
                  "kinship": "ra",
                  "cx": "rx",
                  "xcor": "Erx"
              })
              .drop(columns=["age_diff"]))
df_to_save["Era"] = 0.5 ** df_to_save["DOR"]

(df_to_save[["DOR", "eid", "age", "sex", "eid_rel", "age_rel", "sex_rel", "sex_type", "rel_type", "Era", "ra", "Erx", "rx"]]
 .sort_values(by=["DOR", "eid", "eid_rel"])
 .reset_index(drop=True)
 .to_csv(
     f"{path_to_save}/relatives.info",
     sep='\t',
     index=False
 ))

In [72]:
df_to_save[["DOR", "eid", "age", "sex", "eid_rel", "age_rel", "sex_rel", "sex_type", "rel_type", "Era", "ra", "Erx", "rx"]].head(3)

,DOR,eid,age,sex,eid_rel,age_rel,sex_rel,sex_type,rel_type,Era,ra,Erx,rx
0,3,1000031,52,0,4977365,45,0,ff,nan,0.125,0.1368,0.040435,NaN
3,1,1000094,65,0,3653174,64,0,ff,DS,0.500,0.4942,0.632092,0.75
4,2,1000122,68,0,1390862,63,0,ff,nan,0.250,0.2430,0.501473,NaN


# GS

## Step 1. Load data

In [2]:
df_kinship = pd.read_csv("/data/jerrylee/data/GS/kinpair.xls", sep='\t')
df_pedigree = pd.read_csv("/data/jerrylee/data/GS/pedigree.xls", sep='\t',
                          dtype={"volid": int, "father": int, "mother": int})
df_pheno = pd.read_csv("/data/jerrylee/data/GS/phenotypes.xls", sep='\t')

## Step 2. DOR

### Step 2.1. Check the number of relationships in each DOR

In [3]:
rels = {
    1: ["PC", "SB"],
    2: ["AV", "GP", "HSB"],
    3: ["1C", "HAV", "GG"],
    4: ["H1C"],
}

In [4]:
df_kinship["DOR"] = 0

for dor in rels.keys():
    rels_in_dor = rels[dor]
    df_kinship.loc[df_kinship["rcode"].isin(rels_in_dor), "DOR"] = int(dor)

df_kinship.head()

,volid,relid,rcode,famid,DOR
0,21244,18826,SB,2,1
1,18826,21244,SB,2,1
2,34422,23884,SB,3,1
3,23884,34422,SB,3,1
4,67267,67531,SB,4,1


In [5]:
df_kinship.groupby("DOR").size()

DOR
0    22754
1    36518
2    15592
3     5494
4      150
dtype: int64

### Step 2.2 make info

In [6]:
df_dor123 = df_kinship[df_kinship["DOR"].isin([1, 2, 3])].reset_index(drop=True)

In [7]:
# formatting kinship
before_after = {
    "volid": "eid",
    "relid": "eid_rel",
    "rcode": "rel_type",
}
df_dor123 = df_dor123.rename(columns=before_after)

In [8]:
# formatting info data in df_pheno
df_pheno_info = df_pheno[["id", "age", "sex"]]
before_after_eid = {
    "id": "eid",
}
before_after_eid_rel = {
    "id": "eid_rel",
    "age": "age_rel",
    "sex": "sex_rel",
}

df_p_eid = df_pheno_info.rename(columns=before_after_eid).copy()
df_p_eid_rel = df_pheno_info.rename(columns=before_after_eid_rel).copy()

In [9]:
df_tmp = pd.merge(pd.merge(df_dor123, df_p_eid, on="eid"),
                  df_p_eid_rel,
                  on="eid_rel")
df_tmp["Era"] = 0.5**df_tmp["DOR"]
df_tmp.head(3)

,eid,eid_rel,rel_type,famid,DOR,age,sex,age_rel,sex_rel,Era
0,21244,18826,SB,2,1,36,F,50,F,0.5
1,18826,21244,SB,2,1,50,F,36,F,0.5
2,34422,23884,SB,3,1,33,F,35,M,0.5


In [10]:
# remove duplicat idset
df_tmp['idset'] = df_tmp[['eid', 'eid_rel']].apply(lambda x: '_'.join(map(str, sorted(x))), axis=1)

df_tmp = (df_tmp
          .drop_duplicates(subset="idset", keep="first")
          .drop(columns="idset"))
df_tmp

,eid,eid_rel,rel_type,famid,DOR,age,sex,age_rel,sex_rel,Era
0,21244,18826,SB,2,1,36,F,50,F,0.500
2,34422,23884,SB,3,1,33,F,35,M,0.500
4,67267,67531,SB,4,1,43,F,44,F,0.500
5,79198,67531,PC,4,1,66,F,44,F,0.500
6,20399,67531,SB,4,1,38,F,44,F,0.500
...,...,...,...,...,...,...,...,...,...,...
57584,37545,89552,1C,9006,3,30,F,36,M,0.125
57589,14699,89552,PC,9006,1,60,F,36,M,0.500
57598,79563,18001,SB,9007,1,60,F,66,M,0.500
57599,80945,18001,AV,9007,2,30,F,66,M,0.250


## Step 3. Add X correlation & rel_type annotation

### Step 3.1 sex_type

In [11]:
# sex_type
df_tmp["sex_type"] = "mf"

df_tmp.loc[(df_tmp["sex"] == "M") & (df_tmp["sex_rel"] == "M"), "sex_type"] = "mm"
df_tmp.loc[(df_tmp["sex"] == "F") & (df_tmp["sex_rel"] == "F"), "sex_type"] = "ff"

### Step 3.2 rel_type with rx

In [12]:
# rx corresponding to rel_type
df_tmp["rx"] = np.nan
df_tmp["relationship"] = df_tmp["rel_type"]
df_tmp["relationship"].unique()

array(['SB', 'PC', 'AV', 'GP', '1C', 'HSB', 'HAV', 'GG'], dtype=object)

In [13]:
def flip_and_concat(df: pd.DataFrame, flip_colns: dict):
    flipping_flip_colns = dict((v,k) for k,v in flip_colns.items())
    flip_colns.update(flipping_flip_colns)
    
    
    df_original = df.copy()
    df_flipped = df.copy().rename(columns=flip_colns)
    
    return pd.concat([df_original, df_flipped], axis=0)

In [61]:
df_new = pd.DataFrame()

for famid in [629]: #tqdm(df_tmp["famid"].sample(4)):
    df_pair = df_tmp[(df_tmp["famid"] == famid)]
    df_parent = df_pedigree[df_pedigree["famid"] == famid]
    
    # add sex info in df_parent
    df_ped = (pd.merge(
        df_parent.rename(columns={"volid": "eid"}),
        flip_and_concat(df_pair, {"eid": "eid_rel", "sex": "sex_rel", "age": "age_rel"}),
        on="eid"
    )[["eid", "sex", "father", "mother"]]
    .drop_duplicates(subset="eid"))
    
    df_ped = (df_ped[~((df_ped["father"] == 0) & (df_ped["mother"] == 0))]
              .rename(columns={"eid": "offspring"}))
    
    # sort the younger be "eid" and older be "eid_rel"
    df_pair = ped.sort_id_by_age(df_pair)
    
    # get relationship and rx
    for _, row in df_pair.iterrows():
        ids = row[["eid", "eid_rel"]].to_list()
        if row["DOR"] == 1:
            rel_type, rx = ped.get_rel_rx_dor1(df_ped, 
                                               ids=ids,
                                               relationship=row["relationship"])
        elif row["DOR"] == 2:
            rel_type, rx = ped.get_rel_rx_dor2(df_ped, 
                                               ids=ids,
                                               relationship=row["relationship"])
        elif row["DOR"] == 3:
            # rel_type, rx = ped.get_rel_rx_dor3(df_ped, row["relationship"])
            continue
        else:
            raise Exception("DOR information is inappropriate.")
        
        row["rel_type"] = rel_type
        row["rx"] = rx
        df_new = pd.concat([df_new, row.to_frame().T], axis=0)
    break

df_new

,eid_rel,eid,rel_type,famid,DOR,age_rel,sex,age,sex_rel,Era,sex_type,rx,relationship
0,34570,24766,SM,629,1,50,F,18,M,0.5,mf,0.707107,PC
2,41942,24766,SF,629,1,52,M,18,M,0.5,mm,0,PC
3,5093,24766,None,629,2,46,F,18,M,0.25,mf,NaN,AV
4,92823,34570,DS_fatherSame,629,2,64,F,50,F,0.25,ff,0.5,HSB
5,92823,78069,SM,629,1,64,M,39,F,0.5,mf,0.707107,PC
7,41942,5093,SS_DB,629,1,52,F,46,M,0.5,mf,0.353553,SB


In [56]:
importlib.reload(ped)
ped.test("abc")

abc


In [57]:
pd.DataFrame(row).T

,eid_rel,eid,rel_type,famid,DOR,age_rel,sex,age,sex_rel,Era,sex_type,rx,relationship
7,41942,5093,SS_DB,629,1,52,F,46,M,0.5,mf,0.353553,SB


In [59]:
df_pair

,eid_rel,eid,rel_type,famid,DOR,age_rel,sex,age,sex_rel,Era,sex_type,rx,relationship
0,34570,24766,SM,629,1,50,F,18,M,0.5,mf,0.707107,PC
1,92823,24766,HAV,629,3,64,F,18,M,0.125,mf,NaN,HAV
2,41942,24766,SF,629,1,52,M,18,M,0.5,mm,0,PC
3,5093,24766,AV,629,2,46,F,18,M,0.25,mf,NaN,AV
4,92823,34570,HSB,629,2,64,F,50,F,0.25,ff,NaN,HSB
5,92823,78069,SM,629,1,64,M,39,F,0.5,mf,0.707107,PC
6,34570,78069,HAV,629,3,50,F,39,M,0.125,mf,NaN,HAV
7,41942,5093,SS_DB,629,1,52,F,46,M,0.5,mf,0.353553,SB


In [60]:
ped.get_rel_rx_dor2(df_ped, 
                                               ids=[92823, 34570],
                                               relationship="HSB")

('DS_fatherSame', 0.5)

In [22]:
ids = [5093, 41942]
tmp = df_ped[df_ped["offspring"].isin(ids)]
tmp.loc[tmp["offspring"] == 5093, "sex"].values[0]

'F'

,offspring,sex,father,mother
0,41942,M,82582,99397
2,5093,F,82582,99397
4,78069,M,92518,92823
6,24766,M,41942,34570
10,92823,F,96352,33867
13,34570,F,96352,57536


In [34]:
row[["eid", "eid_rel", "relationship"]]

eid             34570
eid_rel         24766
relationship       PC
Name: 5742, dtype: object

In [39]:
dict_rel_rx = {
    "son-father": ["SF", 0],
    "son-mother": ["SM", 1/np.sqrt(2)],
    "daughter-father": ["DF", 1/np.sqrt(2)],
    "daughter-mother": ["DM", 1/2],
    "son-brother": ["SB", 1/2],
    "son-sister": ["SS_DB", 1/np.sqrt(8)],
    "daughter-sister": ["DS", 3/4],
}

def parent_offspring(df_ped, id1, id2):
    
    for _, row in df_ped.iterrows():
        
        if id1 == row["offspring"]:
            off_sex = row["sex"]
            
            if id2 == row["father"]:
                if off_sex == "M":
                    rel_type, rx = dict_rel_rx["son-father"]
                elif off_sex == "F":
                    rel_type, rx = dict_rel_rx["daughter-father"]
            elif id2 == row["mother"]:
                if off_sex == "M":
                    rel_type, rx = dict_rel_rx["son-mother"]
                elif off_sex == "F":
                    rel_type, rx = dict_rel_rx["daughter-mother"]
            else:
                break
            
            return rel_type, rx
        
        if id2 == row["offspring"]:
            off_sex = row["sex"]
            
            if id1 == row["father"]:
                if off_sex == "M":
                    rel_type, rx = dict_rel_rx["son-father"]
                elif off_sex == "F":
                    rel_type, rx = dict_rel_rx["daughter-father"]
            elif id1 == row["mother"]:
                if off_sex == "M":
                    rel_type, rx = dict_rel_rx["son-mother"]
                elif off_sex == "F":
                    rel_type, rx = dict_rel_rx["daughter-mother"]
            else:
                break
            
            return rel_type, rx

parent_offspring(df_ped, ids[0], ids[1])    

('SM', 0.7071067811865475)

In [36]:
tt[0]

'F'

In [36]:
df_ped

,offspring,sex,father,mother
0,41942,M,82582,99397
2,5093,F,82582,99397
4,78069,M,92518,92823
6,24766,M,41942,34570
10,92823,F,96352,33867
13,34570,F,96352,57536


In [37]:
df_pair

,eid,eid_rel,rel_type,famid,DOR,age,sex,age_rel,sex_rel,Era,sex_type,rx,relationship
5742,34570,24766,PC,629,1,50,F,18,M,0.500,mf,NaN,PC
5743,92823,24766,HAV,629,3,64,F,18,M,0.125,mf,NaN,HAV
5744,41942,24766,PC,629,1,52,M,18,M,0.500,mm,NaN,PC
5745,5093,24766,AV,629,2,46,F,18,M,0.250,mf,NaN,AV
5746,34570,92823,HSB,629,2,50,F,64,F,0.250,ff,NaN,HSB
5748,78069,92823,PC,629,1,39,M,64,F,0.500,mf,NaN,PC
5749,34570,78069,HAV,629,3,50,F,39,M,0.125,mf,NaN,HAV
5755,5093,41942,SB,629,1,46,F,52,M,0.500,mf,NaN,SB


In [ ]:
def find_parent(df_parent: pd.DataFrame, eid: int):
    df_po = df_parent[df_parent["volid"] == eid]
    father_id, mother_id = df_po[["father", "mother"]].values[0]
    return (father_id, mother_id)

def find_sibling(df_parent: pd.DataFrame, fid: int, mid: int):
    df_off = df_parent[(df_parent["father"] == fid) 
                       & (df_parent["mother"] == mid)]
    sib_ids = df_off["volid"].values
    return sib_ids

In [ ]:
# DOR1 = ["PC", "SB"]
def if_pc(row):
    rel_type, rx = None, np.nan
    
    # father-son
    if (
            (row["age"] > row["age_rel"])
            & (row["sex"] == "M") 
            & (row["sex_rel"] == "M")
        ) | (
            (row["age"] < row["age_rel"])
            & (row["sex"] == "M") 
            & (row["sex_rel"] == "M")
        ):
            rel_type, rx = "FS", 0
    
    # mother-son
    elif (
            (row["age"] > row["age_rel"])
            & (row["sex"] == "F") 
            & (row["sex_rel"] == "M")
        ) | (
            (row["age"] < row["age_rel"])
            & (row["sex"] == "M") 
            & (row["sex_rel"] == "F")
        ):
            rel_type, rx = "MS", 1/np.sqrt(2)
    
    # father-daughter
    elif (
            (row["age"] > row["age_rel"])
            & (row["sex"] == "M") 
            & (row["sex_rel"] == "F")
        ) | (
            (row["age"] < row["age_rel"])
            & (row["sex"] == "F") 
            & (row["sex_rel"] == "M")
        ):
            rel_type, rx = "FD", 1/np.sqrt(2)
    
    # mother-daughter
    elif (
            (row["age"] > row["age_rel"])
            & (row["sex"] == "F") 
            & (row["sex_rel"] == "F")
        ) | (
            (row["age"] < row["age_rel"])
            & (row["sex"] == "F") 
            & (row["sex_rel"] == "F")
        ):
            rel_type, rx = "FD", 1/2
    
    return rel_type, rx

#
def if_SB(row):
    sex_mapper = {"M": 1, "F": 0}
    rel_type, rx = None, np.nan
    # brother
    if (
            (row["sex"] == "M") 
            & (row["sex_rel"] == "M")
        ):
            rel_type, rx = "SB", 1/2
    # sister
    elif (
            (row["sex"] == "F") 
            & (row["sex_rel"] == "F")
        ):
            rel_type, rx = "DS", 3/4
    # sibling
    else:
        rel_type, rx = "SS_DB", 1/np.sqrt(8),
    
    return rel_type, rx
        
def get_reltype_and_rx_DOR1(row):
    rel_type, rx = None, np.nan
    
    if row["relationship"] == "PC":
        rel_type, rx = if_pc(row)
    elif row["relationship"] == "SB":
        rel_type, rx = if_SB(row)
    
    return rel_type, rx

In [ ]:
# DOR2 : ["AV", "GP", "HSB"],
def if_HSB(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    
    # same mother or same father?
    father_I, mother_I = find_parent(df_parent, row["eid"])
    father_REL, mother_REL = find_parent(df_parent, row["eid_rel"])
    
    rel_type, rx = None, np.nan
    
    if father_I == father_REL:
        if (
            (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ):
            rel_type, rx = "SB_fatherSame", 0
        
        elif (
            (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ):
            rel_type, rx = "SS_fatherSame", 0
        
        elif (
            (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ):
            rel_type, rx = "DB_fatherSame", 0
    
        elif (
            (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ):
            rel_type, rx = "DS_fatherSame", 1/2
    
    elif mother_I == mother_REL:
        if (
            (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ):
            rel_type, rx = "SB_motherSame", 1/2
        
        elif (
            (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ):
            rel_type, rx = "SS_motherSame", 1/np.sqrt(8)
        
        elif (
            (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ):
            rel_type, rx = "DB_motherSame", 1/np.sqrt(8)
    
        elif (
            (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ):
            rel_type, rx = "DS_motherSame", 1/4
    
    return rel_type, rx
    
def if_GP(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    
    # who is older?
    if row["age"] > row["age_rel"]:
        eid = row["eid_rel"]
        sex_I = row["sex_rel"]
        eid_rel = row["eid"]
    else:
        eid = row["eid"]
        sex_I = row["sex"]
        eid_rel = row["eid_rel"]
        
    # is mother's parents or father's parents?
    father_I, mother_I = find_parent(df_parent, eid)
    father_father, mother_father = find_parent(df_parent, father_I)
    father_mother, mother_mother = find_parent(df_parent, mother_I)
    
    rel_type, rx = None, np.nan
    
    # father's father - father - offspring
    if eid_rel == father_father:
        if sex_I == sex_mapper["M"]:
            rel_type, rx = "SFF", 0
        
        elif sex_I == sex_mapper["F"]:
            rel_type, rx = "DFM", 0
    
    # father's mother - father - offspring
    if eid_rel == mother_father:
        if sex_I == sex_mapper["M"]:
            rel_type, rx = "SFF", 0
        
        elif sex_I == sex_mapper["F"]:
            rel_type, rx = "DFM", 1/2
    
    # mother's father - mother - offspring
    if eid_rel == father_mother:
        if sex_I == sex_mapper["M"]:
            rel_type, rx = "SMF", 1/2
        
        elif sex_I == sex_mapper["F"]:
            rel_type, rx = "DMF", 1/np.sqrt(8)
    
    # mother's mother - mother - offspring
    if eid_rel == mother_mother:
        if sex_I == sex_mapper["M"]:
            rel_type, rx = "SMM", 1/np.sqrt(8)
        
        elif sex_I == sex_mapper["F"]:
            rel_type, rx = "DMM", 1/4
    
    return rel_type, rx

def if_AV(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    
    # who is older?
    if row["age"] > row["age_rel"]:
        eid = row["eid_rel"]
        sex_I = row["sex_rel"]
        eid_rel = row["eid"]
        sex_rel = row["sex"]
    else:
        eid = row["eid"]
        sex_I = row["sex"]
        eid_rel = row["eid_rel"]
        sex_rel = row["sex_rel"]
        
    # is mother's parents or father's parents?
    father_I, mother_I = find_parent(df_parent, eid)
    father_father, mother_father = find_parent(df_parent, father_I)
    father_mother, mother_mother = find_parent(df_parent, mother_I)
    sibs_father = find_sibling(df_parent, father_father, mother_father)
    sibs_mother = find_sibling(df_parent, father_mother, mother_mother)
    
    rel_type, rx = None, np.nan
    
    
    if sex_I == sex_mapper["F"]:
        if eid_rel in sibs_father:
            # daughter-father-brother
            if sex_rel == sex_mapper["M"]:
                rel_type, rx = "DFB", 1/np.sqrt(8)
            # daughter-father-sister
            if sex_rel == sex_mapper["F"]:
                rel_type, rx = "DFS", 1/4
        elif eid_rel in sibs_mother:
            # daughter-mother-brother
            if sex_rel == sex_mapper["M"]:
                rel_type, rx = "DMB", 1/np.sqrt(32)
            # daughter-mother-sister
            if sex_rel == sex_mapper["F"]:
                rel_type, rx = "DMS", 3/8
    
    if sex_I == sex_mapper["M"]:
        if eid_rel in sibs_father:
            # son-father-brother
            if sex_rel == sex_mapper["M"]:
                rel_type, rx = "SFB", 0
            # son-father-sister
            if sex_rel == sex_mapper["F"]:
                rel_type, rx = "SFS", 0
        elif eid_rel in sibs_mother:
            # son-mother-brother
            if sex_rel == sex_mapper["M"]:
                rel_type, rx = "SMB", 1/4
            # son-mother-sister
            if sex_rel == sex_mapper["F"]:
                rel_type, rx = "SMS", 3/np.sqrt(32)

    return rel_type, rx

def get_reltype_and_rx_DOR2(row, df_parent):
    rel_type, rx = None, np.nan
    
    if row["relationship"] == "HSB":
        rel_type, rx = if_HSB(row, df_parent)
    elif row["relationship"] == "GP":
        rel_type, rx = if_GP(row, df_parent)
    elif row["relationship"] == "AV":
        rel_type, rx = if_AV(row, df_parent)
    
    return rel_type, rx

In [ ]:
# DOR3 : ["1C", "HAV", "GG"]
def if_1C(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    rel_type, rx = None, np.nan
    
    # parent's of eid and eid_rel are sibling
    # father 
    
    return rel_type, rx

def if_HAV(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    rel_type, rx = None, np.nan
    
    return rel_type, rx

def if_GG(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    rel_type, rx = None, np.nan
    
    return rel_type, rx

def get_reltype_and_rx_DOR2(row, df_parent):
    rel_type, rx = None, np.nan
    
    if row["relationship"] == "1C":
        rel_type, rx = if_1C(row, df_parent)
    elif row["relationship"] == "HAV":
        rel_type, rx = if_HAV(row, df_parent)
    elif row["relationship"] == "GG":
        rel_type, rx = if_GG(row, df_parent)
    
    return rel_type, rx

In [ ]:
fams_with_rel = df_tmp[df_tmp["relationship"] == "AV"]
(df_tmp[df_tmp["famid"].isin(fams_with_rel["famid"])]
 .groupby("famid")
 .size()
 .to_frame("cnt")
 .query("cnt > 3")
 .sort_values(by="cnt"))

,cnt
famid,
1047,4
4591,4
1270,4
4357,4
4365,4
...,...
3636,109
2902,112
3367,128


In [ ]:
df_new = pd.DataFrame()
for famid in [1047]: #[2137]:#df_tmp["famid"].unique():
    df_pair = df_tmp[(df_tmp["famid"] == famid)]
    df_parent = df_pedigree[df_pedigree["famid"] == famid]
    
    for _, row in df_pair.iterrows():
        rel_type, rx = None, np.nan
        # DOR = 1 (PC, SB)
        if row["DOR"] == 1:
            rel_type, rx = get_reltype_and_rx_DOR1(row)
            
        # DOR = 2 ("AV", "GP", "HSB")
        elif row["DOR"] == 2:
            rel_type, rx = get_reltype_and_rx_DOR2(row, df_parent)
        
        row["rel_type"] = rel_type
        row["rx"] = rx
        df_new = pd.concat([df_new, row.to_frame().T], axis=0)
    break

df_new

,eid,eid_rel,rel_type,famid,DOR,age,sex,age_rel,sex_rel,Era,sex_type,rx,relationship
10216,89887,90827,None,1047,1,50,M,21,M,0.5,mm,NaN,PC
10217,93194,90827,None,1047,2,48,F,21,M,0.25,mf,NaN,AV
10218,68949,90827,None,1047,1,49,F,21,M,0.5,mf,NaN,PC
10221,68949,93194,SS_DB,1047,1,49,F,48,F,0.5,ff,0.353553,SB


In [ ]:
df_parent

,famid,volid,father,mother
5868,1047,89887,0,0
5869,1047,95556,0,0
5870,1047,67845,0,0
5871,1047,90827,89887,68949
5872,1047,68949,67845,95556
5873,1047,93194,67845,95556


In [ ]:
df_pair

,eid,eid_rel,rel_type,famid,DOR,age,sex,age_rel,sex_rel,Era,sex_type,rx,relationship
21472,76060,78946,PC,2137,1,63,F,33,F,0.50,ff,NaN,PC
21473,63482,78946,HSB,2137,2,27,M,33,F,0.25,mf,NaN,HSB
21474,76060,63482,PC,2137,1,63,F,27,M,0.50,mf,NaN,PC
21476,92997,63482,PC,2137,1,65,M,27,M,0.50,mm,NaN,PC


In [86]:
def flip_and_concat(df: pd.DataFrame, flip_colns: dict):
    flipping_flip_colns = dict((v,k) for k,v in flip_colns.items())
    flip_colns.update(flipping_flip_colns)
    
    
    df_original = df.copy()
    df_flipped = df.copy().rename(columns=flip_colns)
    
    return pd.concat([df_original, df_flipped], axis=0)

def get_edges(df_ped: pd.DataFrame):
    edges = []
    for _, row in df_ped.iterrows():
        edges.append([row["offspring"], row["father"]])
        edges.append([row["offspring"], row["mother"]])

    return edges

In [95]:
df_new = pd.DataFrame()

for famid in [629]: #tqdm(df_tmp["famid"].sample(4)):
    df_pair = df_tmp[(df_tmp["famid"] == famid)]
    df_parent = df_pedigree[df_pedigree["famid"] == famid]
    
    # add sex info in df_parent
    df_ped = (pd.merge(
        df_parent.rename(columns={"volid": "eid"}),
        flip_and_concat(df_pair, {"eid": "eid_rel", "sex": "sex_rel", "age": "age_rel"}),
        on="eid"
    )[["eid", "sex", "father", "mother"]]
    .drop_duplicates(subset="eid"))
    
    df_ped = (df_ped[~((df_ped["father"] == 0) & (df_ped["mother"] == 0))]
              .rename(columns={"eid": "offspring"}))
    edges = get_edges(df_ped)
    break
    # find path
    for _, row in df_pair.iterrows():
        rel_type, rx = None, np.nan
        
        # paths = find_paths(df_ped, row["eid"], row["eid_rel"])
        
        i = 0
        rels = []
        rxs = 1
        good = True
        break
    break
    

In [96]:
node1 = 41942
node2 = 5093

print(find_all_paths(edges, node1, node2))

[]


In [99]:
df_pair

,eid,eid_rel,rel_type,famid,DOR,age,sex,age_rel,sex_rel,Era,sex_type,rx,relationship
5742,34570,24766,PC,629,1,50,F,18,M,0.500,mf,NaN,PC
5743,92823,24766,HAV,629,3,64,F,18,M,0.125,mf,NaN,HAV
5744,41942,24766,PC,629,1,52,M,18,M,0.500,mm,NaN,PC
5745,5093,24766,AV,629,2,46,F,18,M,0.250,mf,NaN,AV
5746,34570,92823,HSB,629,2,50,F,64,F,0.250,ff,NaN,HSB
5748,78069,92823,PC,629,1,39,M,64,F,0.500,mf,NaN,PC
5749,34570,78069,HAV,629,3,50,F,39,M,0.125,mf,NaN,HAV
5755,5093,41942,SB,629,1,46,F,52,M,0.500,mf,NaN,SB


In [100]:
import pandas as pd
import networkx as nx

# Define the dataframe
data = {
    'offspring': [41942, 5093, 78069, 24766, 92823, 34570],
    'father': [82582, 82582, 92518, 41942, 96352, 96352],
    'mother': [99397, 99397, 92823, 34570, 33867, 57536]
}
df = pd.DataFrame(data)

# Create a directed graph
G = nx.DiGraph()

# Add edges from father to offspring and from mother to offspring
for _, row in df.iterrows():
    G.add_edge(row['father'], row['offspring'])
    G.add_edge(row['mother'], row['offspring'])

# Function to find the relationship between two individuals
def find_relationship(person1, person2):
    ancestors_p1 = nx.ancestors(G, person1)
    ancestors_p2 = nx.ancestors(G, person2)
    
    common_ancestors = ancestors_p1.intersection(ancestors_p2)
    
    if person1 in ancestors_p2 or person2 in ancestors_p1:
        return "Parent-Offspring Relationship"
    elif len(common_ancestors) == 0:
        return "Unrelated"
    elif len(common_ancestors) == 1:
        ancestor = common_ancestors.pop()
        if G.has_edge(ancestor, person1) and G.has_edge(ancestor, person2):
            return "Full-Sibling Relationship"
        elif (G.has_edge(ancestor, person1) and not G.has_edge(ancestor, person2)) or \
             (not G.has_edge(ancestor, person1) and G.has_edge(ancestor, person2)):
            return "Half-Sibling Relationship"
        elif (person1 in G.successors(ancestor) and person2 in G.successors(ancestor)) or \
             (person1 in G.predecessors(ancestor) and person2 in G.predecessors(ancestor)):
            return "Avuncular Relationship"
        elif (person1 in G.predecessors(ancestor) and person2 in ancestors_p1) or \
             (person2 in G.predecessors(ancestor) and person1 in ancestors_p2):
            return "Grandparent-Offspring Relationship"
    elif len(common_ancestors) == 2:
        ancestor1, ancestor2 = common_ancestors
        if (G.has_edge(ancestor1, person1) and G.has_edge(ancestor2, person2)) or \
           (G.has_edge(ancestor2, person1) and G.has_edge(ancestor1, person2)):
            return "First Cousin Relationship"
        elif (G.has_edge(ancestor1, person1) and not G.has_edge(ancestor1, person2) and \
              G.has_edge(ancestor2, person2)) or \
             (G.has_edge(ancestor2, person1) and not G.has_edge(ancestor2, person2) and \
              G.has_edge(ancestor1, person2)):
            return "Half-Avuncular Relationship"
    elif len(common_ancestors) == 3:
        return "Great-Grandparent-Offspring Relationship"
    else:
        return "Complex Relationship"

# Example relationships
print(find_relationship(41942, 24766))  # Parent-Offspring Relationship
print(find_relationship(5093, 41942))   # Full-Sibling Relationship
print(find_relationship(34570, 92823))  # Half-Sibling Relationship
print(find_relationship(92823, 24766))  # Avuncular Relationship


ModuleNotFoundError: No module named 'networkx'

In [85]:
edges = []
for _, row in df_ped.iterrows():
    edges.append([row["offspring"], row["father"]])
    edges.append([row["offspring"], row["mother"]])
    break


edges

[[41942, 82582], [41942, 99397]]

In [56]:
dict_rel_rx = {
    "son-father": ["SF", 0],
    "son-mother": ["SM", 1/np.sqrt(2)],
    "daughter-father": ["DF", 1/np.sqrt(2)],
    "daughter-mother": ["DM", 1/2],
    "son-brother": ["SB", 1/2],
    "son-sister": ["SS_DB", 1/np.sqrt(8)],
    "daughter-sister": ["DS", 3/4],
}

def if_PC(df_ped, ii_i, ii_r):
    rel_type, rx = None, np.nan
    
    dict_rel_rx = {
        "son-father": ["SF", 0],
        "son-mother": ["SM", 1/np.sqrt(2)],
        "daughter-father": ["DF", 1/np.sqrt(2)],
        "daughter-mother": ["DM", 1/2],
        "son-brother": ["SB", 1/2],
        "son-sister": ["SS_DB", 1/np.sqrt(8)],
        "daughter-sister": ["DS", 3/4],
    }
    
    def find_parent(df: pd.DataFrame, eid: int):
        df_po = df[df["offspring"] == eid]
        father_id, mother_id = df_po[["father", "mother"]].values[0]
        return (father_id, mother_id)
    
    relationship = None
    
    if ii_i in df_ped["offspring"].to_list():
        sex_i = df_ped.loc[df_ped["offspring"] == ii_i, "sex"].values[0]
        father_id, mother_id = find_parent(df_ped, ii_i)
        if (father_id == ii_r) & (sex_i == "M"):
            relationship = "son-father"
        if (mother_id == ii_r) & (sex_i == "M"):
            relationship = "son-mother"
        if (father_id == ii_r) & (sex_i == "F"):
            relationship = "daughter-father"
        if (mother_id == ii_r) & (sex_i == "F"):
            relationship = "daughter-mother"
    
    if ii_r in df_ped["offspring"].to_list():
        sex_r = df_ped.loc[df_ped["offspring"] == ii_r, "sex"].values[0]
        father_id, mother_id = find_parent(df_ped, ii_r)
        if (father_id == ii_i) & (sex_r == "M"):
            relationship = "son-father"
        if (mother_id == ii_i) & (sex_r == "M"):
            relationship = "son-mother"
        if (father_id == ii_i) & (sex_r == "F"):
            relationship = "daughter-father"
        if (mother_id == ii_i) & (sex_r == "F"):
            relationship = "daughter-mother"
    
    return dict_rel_rx[relationship]
        
def if_SB(sex, sex_rel):
    rel_type, rx = None, np.nan
    
    dict_rel_rx = {
        "son-father": ["SF", 0],
        "son-mother": ["SM", 1/np.sqrt(2)],
        "daughter-father": ["DF", 1/np.sqrt(2)],
        "daughter-mother": ["DM", 1/2],
        "son-brother": ["SB", 1/2],
        "son-sister": ["SS_DB", 1/np.sqrt(8)],
        "daughter-sister": ["DS", 3/4],
    }
    
    # brother
    if (
            (sex == "M") 
            & (sex_rel == "M")
        ):
            rel_type, rx = dict_rel_rx["son-brother"]
    # sister
    elif (
            (sex == "F") 
            & (sex_rel == "F")
        ):
            rel_type, rx = dict_rel_rx["daughter-sister"]
    # sibling
    else:
        rel_type, rx = dict_rel_rx["son-sister"]
    
    return rel_type, rx
        
# def if_pc(age, sex, age_rel, sex_rel):
#     rel_type, rx = None, np.nan
    
#     # father-son
#     if (
#             (age > age_rel)
#             & (sex == "M") 
#             & (sex_rel == "M")
#         ) | (
#             (age < age_rel)
#             & (sex == "M") 
#             & (sex_rel == "M")
#         ):
#             rel_type, rx = dict_rel_rx["son-father"]
    
#     # mother-son
#     elif (
#             (age > age_rel)
#             & (sex == "F") 
#             & (sex_rel == "M")
#         ) | (
#             (age < age_rel)
#             & (sex == "M") 
#             & (sex_rel == "F")
#         ):
#             rel_type, rx = dict_rel_rx["son-mother"]
    
#     # father-daughter
#     elif (
#             (age > age_rel)
#             & (sex == "M") 
#             & (sex_rel == "F")
#         ) | (
#             (age < age_rel)
#             & (sex == "F") 
#             & (sex_rel == "M")
#         ):
#             rel_type, rx = dict_rel_rx["daughter-father"]
    
#     # mother-daughter
#     elif (
#             (age > age_rel)
#             & (sex == "F") 
#             & (sex_rel == "F")
#         ) | (
#             (age < age_rel)
#             & (sex == "F") 
#             & (sex_rel == "F")
#         ):
#             rel_type, rx = dict_rel_rx["daughter-mother"]
    
#     return rel_type, rx

# def if_SB(sex, sex_rel):
#     rel_type, rx = None, np.nan
#     # brother
#     if (
#             (sex == "M") 
#             & (sex_rel == "M")
#         ):
#             rel_type, rx = dict_rel_rx["son-brother"]
#     # sister
#     elif (
#             (sex == "F") 
#             & (sex_rel == "F")
#         ):
#             rel_type, rx = dict_rel_rx["daughter-sister"]
#     # sibling
#     else:
#         rel_type, rx = dict_rel_rx["son-sister"]
    
#     return rel_type, rx

# def get_sex_age(df_pair, eid, eid_rel):
#     tmp = df_pair[
#         (df_pair["eid"] == eid)
#         & (df_pair["eid_rel"] == eid_rel)
#     ]
#     tmp_reverse = df_pair[
#         (df_pair["eid"] == eid_rel)
#         & (df_pair["eid_rel"] == eid)
#     ]
    
#     try:
#         if len(tmp) > 0:
#             sex = tmp["sex"].values[0]
#             age = tmp["age"].values[0]
#             sex_rel = tmp["sex_rel"].values[0]
#             age_rel = tmp["age_rel"].values[0]
#         else:
#             sex = tmp_reverse["sex_rel"].values[0]
#             age = tmp_reverse["age_rel"].values[0]
#             sex_rel = tmp_reverse["sex"].values[0]
#             age_rel = tmp_reverse["age"].values[0]
#     except:
#         return None, None, None, None
    
#     return sex, age, sex_rel, age_rel

In [57]:
def find_path(df, eid, eid_rel):
    from collections import defaultdict, deque
    
    parent_to_child = defaultdict(list)
    child_to_parents = {}
    
    for _, row in df.iterrows():
        offspring, father, mother = row['offspring'], row['father'], row['mother']
        parent_to_child[father].append(offspring)
        parent_to_child[mother].append(offspring)
        child_to_parents[offspring] = (father, mother)

    if eid_rel in parent_to_child.get(eid, []):
        return [eid, eid_rel]
    if eid in child_to_parents and eid_rel in child_to_parents[eid]:
        return [eid, eid_rel]

    queue = deque([(eid, [eid])])
    visited = set()

    while queue:
        current, path = queue.popleft()
        
        if current == eid_rel:
            return path
        
        if current in visited:
            continue
        visited.add(current)
        
        if current in child_to_parents:
            father, mother = child_to_parents[current]
            parents = [father, mother]

            # Enqueue the parents
            for parent in parents:
                if parent not in visited:
                    queue.append((parent, path + [parent]))

            # Enqueue siblings and their children
            siblings = set(parent_to_child[father]) | set(parent_to_child[mother]) - {current}
            for sibling in siblings:
                if sibling not in visited:
                    if child_to_parents.get(sibling) == (father, mother):
                        sibling_path = path + [[father, mother], sibling]
                    else:
                        sibling_path = path + [parent, sibling]

                    queue.append((sibling, sibling_path))
                    # Also consider the children of siblings
                    for offspring in parent_to_child[sibling]:
                        if offspring not in visited:
                            queue.append((offspring, sibling_path + [offspring]))

        # Enqueue the children
        if current in parent_to_child:
            for child in parent_to_child[current]:
                if child not in visited:
                    queue.append((child, path + [child]))

    return "No path found"

In [58]:
def get_sex(df_ped, eid):
    """
    columns = [offspring	father	mother	sex]
    """
    if eid in df_ped["offspring"].to_list():
        return df_ped.loc[df_ped["offspring"] == eid, "sex"].values[0]
    elif eid in df_ped["father"].to_list():
        return "M"
    elif eid in df_ped["mother"].to_list():
        return "F"
    else:
        return None

In [61]:
df_new = pd.DataFrame()

for famid in tqdm(df_tmp["famid"].unique()):
    df_pair = df_tmp[(df_tmp["famid"] == famid)]
    df_pair_sex = (pd.concat(
        [
            df_pair[["eid", "sex"]], 
            df_pair[["eid_rel", "sex_rel"]].rename(
                columns={
                    "eid_rel": "eid",
                    "sex_rel": "sex"
                }
            ),
        ], 
        axis=0
        )
        .drop_duplicates(subset="eid")
    )
    df_parent = df_pedigree[df_pedigree["famid"] == famid]
    
    df_ped = pd.merge((df_parent
                       .drop(columns="famid")
                       .rename(columns={"volid": "eid"})),
                      df_pair_sex,
                      on="eid")
    df_ped = (df_ped[~((df_ped["father"] == 0) & (df_ped["mother"] == 0))]
              .rename(columns={"eid": "offspring"}))
    
    for _, row in df_pair.iterrows():
        rel_type, rx = None, np.nan
        
        path = find_path(df_ped, row["eid"], row["eid_rel"])
        
        i = 0
        rels = []
        rxs = 1
        good = True
        for j in range(len(path)):
            target_path = path[i:i+3]
            
            # stopping condition
            if len(target_path) < 2:
                break
            
            ii_i = target_path[0]
            sex_i = get_sex(df_ped, ii_i)
            if isinstance(target_path[1], list):
                # full-sibling case
                ii_r = target_path[2]
                sex_r = get_sex(df_ped, ii_r)
                
                if (sex_i == None) | (sex_r == None):
                    good = False
                    break
                
                rel_type, rx = if_SB(sex_i, sex_r)
                rels.append(rel_type)
                rxs *= rx
                i += 2
            else:
                # parent-offspring case
                ii_r = target_path[1]
                sex_r = get_sex(df_ped, ii_r)
                
                if (sex_i == None) | (sex_r == None):
                    good = False
                    break
                
                rel_type, rx = if_PC(df_ped, ii_i, ii_r)
                rels.append(rel_type)
                rxs *= rx
                i += 1
                
            if not good:
                continue
        if good: 
            row["rel_type"] = "_".join(rels)
            row["rx"] = rxs
            df_new = pd.concat([df_new, row.to_frame().T], axis=0)

 10%|█         | 482/4757 [00:14<02:10, 32.84it/s]


KeyError: None

In [69]:
df_ped

,offspring,father,mother,sex
0,41942,82582,99397,M
1,5093,82582,99397,F
2,78069,92518,92823,M
3,24766,41942,34570,M
4,92823,96352,33867,F
5,34570,96352,57536,F


In [67]:
# current relationship
row["eid"], row["eid_rel"], path

(92823, 24766, [92823, 33867, 34570, 24766])

In [68]:
# iterative path
ii_i, ii_r, target_path

(33867, 34570, [33867, 34570, 24766])

In [66]:
path

[92823, 33867, 34570, 24766]

In [52]:
df_ped

,offspring,father,mother,sex
2,20910,92612,14699,F
3,89552,92612,14699,M
4,25938,44272,16339,M
5,37545,44272,16339,F
6,16339,61106,51388,F
7,92612,61106,51388,M


In [53]:
ii_i, ii_r

(37545, 16339)

In [40]:
df_new = pd.DataFrame()

for famid in tqdm(df_tmp["famid"].unique()):
    df_pair = df_tmp[(df_tmp["famid"] == famid)]
    df_parent = df_pedigree[df_pedigree["famid"] == famid]
    
    for _, row in df_pair.iterrows():
        rel_type, rx = None, np.nan
        
        ped_tree = (
            df_parent[(df_parent["father"] != 0) & (df_parent["mother"] != 0)]
            .drop(columns="famid")
            .rename(columns={"volid": "offspring"})
            .copy()
        )
        path = find_path(ped_tree, row["eid"], row["eid_rel"])
        
        i = 0
        rels = []
        rxs = 1
        good = True
        for j in range(len(path)):
            target_path = path[i:i+3]
            
            # stopping condition
            if len(target_path) < 2:
                break
            
            if isinstance(target_path[1], list):
                # full-sibling case
                ii_i = target_path[0]
                ii_r = target_path[2]
                sex, age, sex_rel, age_rel = get_sex_age(
                    df_pair, ii_i, ii_r,
                )
                
                if sex == None:
                    good = False
                    break
                
                rel_type, rx = if_SB(sex, sex_rel)
                rels.append(rel_type)
                rxs *= rx
                i += 2
            else:
                # parent-offspring case
                sex, age, sex_rel, age_rel = get_sex_age(
                    df_pair,
                    target_path[0], 
                    target_path[1]
                )
                
                if sex == None:
                    good = False
                    break
                
                rel_type, rx = if_pc(age, sex, age_rel, sex_rel)
                rels.append(rel_type)
                rxs *= rx
                i += 1
                
            if not good:
                continue
            
        if good: 
            row["rel_type"] = "_".join(rels)
            row["rx"] = rxs
            df_new = pd.concat([df_new, row.to_frame().T], axis=0)

  0%|          | 0/4757 [00:00<?, ?it/s]

  1%|          | 54/4757 [00:03<04:56, 15.85it/s]


KeyboardInterrupt: 

In [377]:
df_new

,eid,eid_rel,rel_type,famid,DOR,age,sex,age_rel,sex_rel,Era,sex_type,rx,relationship
0,21244,18826,DS,2,1,36,F,50,F,0.5,ff,0.75,SB
2,34422,23884,SS_DB,3,1,33,F,35,M,0.5,mf,0.353553,SB
4,67267,67531,DS,4,1,43,F,44,F,0.5,ff,0.75,SB
5,79198,67531,DM,4,1,66,F,44,F,0.5,ff,0.5,PC
6,20399,67531,DS,4,1,38,F,44,F,0.5,ff,0.75,SB
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57584,37545,89552,DM_SS_DB_SF,9006,3,30,F,36,M,0.125,mf,0.0,1C
57589,14699,89552,SM,9006,1,60,F,36,M,0.5,mf,0.707107,PC
57598,79563,18001,SS_DB,9007,1,60,F,66,M,0.5,mf,0.353553,SB
57599,80945,18001,DM_SS_DB,9007,2,30,F,66,M,0.25,mf,0.176777,AV


In [378]:
df_tmp

,eid,eid_rel,rel_type,famid,DOR,age,sex,age_rel,sex_rel,Era,sex_type,rx,relationship
0,21244,18826,SB,2,1,36,F,50,F,0.500,ff,NaN,SB
2,34422,23884,SB,3,1,33,F,35,M,0.500,mf,NaN,SB
4,67267,67531,SB,4,1,43,F,44,F,0.500,ff,NaN,SB
5,79198,67531,PC,4,1,66,F,44,F,0.500,ff,NaN,PC
6,20399,67531,SB,4,1,38,F,44,F,0.500,ff,NaN,SB
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57584,37545,89552,1C,9006,3,30,F,36,M,0.125,mf,NaN,1C
57589,14699,89552,PC,9006,1,60,F,36,M,0.500,mf,NaN,PC
57598,79563,18001,SB,9007,1,60,F,66,M,0.500,mf,NaN,SB
57599,80945,18001,AV,9007,2,30,F,66,M,0.250,mf,NaN,AV


In [22]:
def find_parent(df_parent: pd.DataFrame, eid: int):
    df_po = df_parent[df_parent["volid"] == eid]
    father_id, mother_id = df_po[["father", "mother"]].values[0]
    return (father_id, mother_id)

def find_sibling(df_parent: pd.DataFrame, fid: int, mid: int):
    df_off = df_parent[(df_parent["father"] == fid) 
                       & (df_parent["mother"] == mid)]
    sib_ids = df_off["volid"].values
    return sib_ids

In [23]:
# DOR1 = ["PC", "SB"]
def if_pc(row):
    sex_mapper = {"M": 1, "F": 0}
    rel_type, rx = None, np.nan
    
    # father-son
    if (
            (row["age"] > row["age_rel"])
            & (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ) | (
            (row["age"] < row["age_rel"])
            & (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ):
            rel_type, rx = "FS", 0
    
    # mother-son
    elif (
            (row["age"] > row["age_rel"])
            & (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ) | (
            (row["age"] < row["age_rel"])
            & (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ):
            rel_type, rx = "MS", 1/np.sqrt(2)
    
    # father-daughter
    elif (
            (row["age"] > row["age_rel"])
            & (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ) | (
            (row["age"] < row["age_rel"])
            & (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ):
            rel_type, rx = "FD", 1/np.sqrt(2)
    
    # mother-daughter
    elif (
            (row["age"] > row["age_rel"])
            & (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ) | (
            (row["age"] < row["age_rel"])
            & (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ):
            rel_type, rx = "FD", 1/2
    
    return rel_type, rx

#
def if_SB(row):
    sex_mapper = {"M": 1, "F": 0}
    rel_type, rx = None, np.nan
    # brother
    if (
            (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ):
            rel_type, rx = "SB", 1/2
    # sister
    elif (
            (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ):
            rel_type, rx = "DS", 3/4
    # sibling
    else:
        rel_type, rx = "SS_DB", 1/np.sqrt(8),
    
    return rel_type, rx
        
def get_reltype_and_rx_DOR1(row):
    rel_type, rx = None, np.nan
    
    if row["relationship"] == "PC":
        rel_type, rx = if_pc(row)
    elif row["relationship"] == "SB":
        rel_type, rx = if_SB(row)
    
    return rel_type, rx

In [24]:
# DOR2 : ["AV", "GP", "HSB"],
def if_HSB(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    
    # same mother or same father?
    father_I, mother_I = find_parent(df_parent, row["eid"])
    father_REL, mother_REL = find_parent(df_parent, row["eid_rel"])
    
    rel_type, rx = None, np.nan
    
    if father_I == father_REL:
        if (
            (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ):
            rel_type, rx = "SB_fatherSame", 0
        
        elif (
            (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ):
            rel_type, rx = "SS_fatherSame", 0
        
        elif (
            (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ):
            rel_type, rx = "DB_fatherSame", 0
    
        elif (
            (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ):
            rel_type, rx = "DS_fatherSame", 1/2
    
    elif mother_I == mother_REL:
        if (
            (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ):
            rel_type, rx = "SB_motherSame", 1/2
        
        elif (
            (row["sex"] == sex_mapper["M"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ):
            rel_type, rx = "SS_motherSame", 1/np.sqrt(8)
        
        elif (
            (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["M"])
        ):
            rel_type, rx = "DB_motherSame", 1/np.sqrt(8)
    
        elif (
            (row["sex"] == sex_mapper["F"]) 
            & (row["sex_rel"] == sex_mapper["F"])
        ):
            rel_type, rx = "DS_motherSame", 1/4
    
    return rel_type, rx
    
def if_GP(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    
    # who is older?
    if row["age"] > row["age_rel"]:
        eid = row["eid_rel"]
        sex_I = row["sex_rel"]
        eid_rel = row["eid"]
    else:
        eid = row["eid"]
        sex_I = row["sex"]
        eid_rel = row["eid_rel"]
        
    # is mother's parents or father's parents?
    father_I, mother_I = find_parent(df_parent, eid)
    father_father, mother_father = find_parent(df_parent, father_I)
    father_mother, mother_mother = find_parent(df_parent, mother_I)
    
    rel_type, rx = None, np.nan
    
    # father's father - father - offspring
    if eid_rel == father_father:
        if sex_I == sex_mapper["M"]:
            rel_type, rx = "SFF", 0
        
        elif sex_I == sex_mapper["F"]:
            rel_type, rx = "DFM", 0
    
    # father's mother - father - offspring
    if eid_rel == mother_father:
        if sex_I == sex_mapper["M"]:
            rel_type, rx = "SFF", 0
        
        elif sex_I == sex_mapper["F"]:
            rel_type, rx = "DFM", 1/2
    
    # mother's father - mother - offspring
    if eid_rel == father_mother:
        if sex_I == sex_mapper["M"]:
            rel_type, rx = "SMF", 1/2
        
        elif sex_I == sex_mapper["F"]:
            rel_type, rx = "DMF", 1/np.sqrt(8)
    
    # mother's mother - mother - offspring
    if eid_rel == mother_mother:
        if sex_I == sex_mapper["M"]:
            rel_type, rx = "SMM", 1/np.sqrt(8)
        
        elif sex_I == sex_mapper["F"]:
            rel_type, rx = "DMM", 1/4
    
    return rel_type, rx

def if_AV(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    
    # who is older?
    if row["age"] > row["age_rel"]:
        eid = row["eid_rel"]
        sex_I = row["sex_rel"]
        eid_rel = row["eid"]
        sex_rel = row["sex"]
    else:
        eid = row["eid"]
        sex_I = row["sex"]
        eid_rel = row["eid_rel"]
        sex_rel = row["sex_rel"]
        
    # is mother's parents or father's parents?
    father_I, mother_I = find_parent(df_parent, eid)
    father_father, mother_father = find_parent(df_parent, father_I)
    father_mother, mother_mother = find_parent(df_parent, mother_I)
    sibs_father = find_sibling(df_parent, father_father, mother_father)
    sibs_mother = find_sibling(df_parent, father_mother, mother_mother)
    
    rel_type, rx = None, np.nan
    
    
    if sex_I == sex_mapper["F"]:
        if eid_rel in sibs_father:
            # daughter-father-brother
            if sex_rel == sex_mapper["M"]:
                rel_type, rx = "DFB", 1/np.sqrt(8)
            # daughter-father-sister
            if sex_rel == sex_mapper["F"]:
                rel_type, rx = "DFS", 1/4
        elif eid_rel in sibs_mother:
            # daughter-mother-brother
            if sex_rel == sex_mapper["M"]:
                rel_type, rx = "DMB", 1/np.sqrt(32)
            # daughter-mother-sister
            if sex_rel == sex_mapper["F"]:
                rel_type, rx = "DMS", 3/8
    
    if sex_I == sex_mapper["M"]:
        if eid_rel in sibs_father:
            # son-father-brother
            if sex_rel == sex_mapper["M"]:
                rel_type, rx = "SFB", 0
            # son-father-sister
            if sex_rel == sex_mapper["F"]:
                rel_type, rx = "SFS", 0
        elif eid_rel in sibs_mother:
            # son-mother-brother
            if sex_rel == sex_mapper["M"]:
                rel_type, rx = "SMB", 1/4
            # son-mother-sister
            if sex_rel == sex_mapper["F"]:
                rel_type, rx = "SMS", 3/np.sqrt(32)

    return rel_type, rx

def get_reltype_and_rx_DOR2(row, df_parent):
    rel_type, rx = None, np.nan
    
    if row["relationship"] == "HSB":
        rel_type, rx = if_HSB(row, df_parent)
    elif row["relationship"] == "GP":
        rel_type, rx = if_GP(row, df_parent)
    elif row["relationship"] == "AV":
        rel_type, rx = if_AV(row, df_parent)
    
    return rel_type, rx

In [25]:
# DOR3 : ["1C", "HAV", "GG"]
def if_1C(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    rel_type, rx = None, np.nan
    
    # parent's of eid and eid_rel are sibling
    # father 
    
    return rel_type, rx

def if_HAV(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    rel_type, rx = None, np.nan
    
    return rel_type, rx

def if_GG(row, df_parent):
    sex_mapper = {"M": 1, "F": 0}
    rel_type, rx = None, np.nan
    
    return rel_type, rx

def get_reltype_and_rx_DOR2(row, df_parent):
    rel_type, rx = None, np.nan
    
    if row["relationship"] == "1C":
        rel_type, rx = if_1C(row, df_parent)
    elif row["relationship"] == "HAV":
        rel_type, rx = if_HAV(row, df_parent)
    elif row["relationship"] == "GG":
        rel_type, rx = if_GG(row, df_parent)
    
    return rel_type, rx

In [39]:
fams_with_rel = df_tmp[df_tmp["relationship"] == "AV"]
(df_tmp[df_tmp["famid"].isin(fams_with_rel["famid"])]
 .groupby("famid")
 .size()
 .to_frame("cnt")
 .query("cnt > 3")
 .sort_values(by="cnt"))

,cnt
famid,
1047,4
4591,4
1270,4
4357,4
4365,4
...,...
3636,109
2902,112
3367,128


In [41]:
df_new = pd.DataFrame()
for famid in [1047]: #[2137]:#df_tmp["famid"].unique():
    df_pair = df_tmp[(df_tmp["famid"] == famid)]
    df_parent = df_pedigree[df_pedigree["famid"] == famid]
    
    for _, row in df_pair.iterrows():
        rel_type, rx = None, np.nan
        # DOR = 1 (PC, SB)
        if row["DOR"] == 1:
            rel_type, rx = get_reltype_and_rx_DOR1(row)
            
        # DOR = 2 ("AV", "GP", "HSB")
        elif row["DOR"] == 2:
            rel_type, rx = get_reltype_and_rx_DOR2(row, df_parent)
        
        row["rel_type"] = rel_type
        row["rx"] = rx
        df_new = pd.concat([df_new, row.to_frame().T], axis=0)
    break

df_new

,eid,eid_rel,rel_type,famid,DOR,age,sex,age_rel,sex_rel,Era,sex_type,rx,relationship
10216,89887,90827,None,1047,1,50,M,21,M,0.5,mm,NaN,PC
10217,93194,90827,None,1047,2,48,F,21,M,0.25,mf,NaN,AV
10218,68949,90827,None,1047,1,49,F,21,M,0.5,mf,NaN,PC
10221,68949,93194,SS_DB,1047,1,49,F,48,F,0.5,ff,0.353553,SB


In [42]:
df_parent

,famid,volid,father,mother
5868,1047,89887,0,0
5869,1047,95556,0,0
5870,1047,67845,0,0
5871,1047,90827,89887,68949
5872,1047,68949,67845,95556
5873,1047,93194,67845,95556


In [38]:
df_pair

,eid,eid_rel,rel_type,famid,DOR,age,sex,age_rel,sex_rel,Era,sex_type,rx,relationship
21472,76060,78946,PC,2137,1,63,F,33,F,0.50,ff,NaN,PC
21473,63482,78946,HSB,2137,2,27,M,33,F,0.25,mf,NaN,HSB
21474,76060,63482,PC,2137,1,63,F,27,M,0.50,mf,NaN,PC
21476,92997,63482,PC,2137,1,65,M,27,M,0.50,mm,NaN,PC


In [ ]:
[["DOR", "eid", "age", "sex", "eid_rel", "age_rel", "sex_rel", "sex_type", "rel_type", "Era", "ra", "Erx", "rx"]]